<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">پاسخ آینده را از جدول خارج کنید</h1>
<p style="text-align:right">درس 39 از 92 · چرا مدل نباید پاسخ را از آینده بردارد؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">33-mask</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-03/33-mask.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right"><bdi dir="ltr">Mask</bdi> علّی را پیش از <bdi dir="ltr">Softmax</bdi> اعمال کنید و قطر و مجموع سطرها را بسنجید.</p><p style="text-align:right"><span class="phrase-lead" style="white-space:nowrap">پیش‌نیاز: محورهای</span> <bdi dir="ltr">Query/Key</bdi> و <bdi dir="ltr">Softmax</bdi> سطری را بشناسید.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۶۰–۱۰۵ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">در جدول سه‌موقعیتی، سطر صفر چند <bdi dir="ltr">Key</bdi> مجاز دارد؟ اگر امتیاز آینده را صفر کنیم، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">exp</code> آن صفر می‌شود؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
scores = torch.tensor([[1.,2.,3.],[4.,2.,0.],[-1.,0.,1.]])
print('unmasked weights:',scores.softmax(-1))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">causal_weights(scores)</code> برای <bdi dir="ltr">Tensor</bdi> مربعی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(...,T,T)</code> بنویسید. قطر و پایین آن مجازند؛ خانه‌های دیگر پیش از <bdi dir="ltr">Softmax</bdi> به <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">-inf</code> بروند. خروجی وزن‌ها باشد و ورودی را درجا تغییر ندهید.</p>
</div>

In [ ]:
def causal_weights(scores):
    # TODO
    return None

In [ ]:
def test_exercise():
    original = scores.clone()
    result = causal_weights(scores)
    if result is None: return False
    torch.testing.assert_close(result[0],torch.tensor([1.,0.,0.]))
    assert torch.count_nonzero(result.triu(1)) == 0
    torch.testing.assert_close(result.sum(-1),torch.ones(3))
    assert torch.equal(scores,original)
    for T in (1,4):
        s = torch.zeros(2,3,T,T)
        w = causal_weights(s)
        torch.testing.assert_close(w.sum(-1),torch.ones(2,3,T))
        torch.testing.assert_close(w[..., -1,:],torch.full((2,3,T),1/T))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">تابع آمادهٔ <bdi dir="ltr">PyTorch</bdi> با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">is_causal=True</code> محدودیت علّی را درون خودش اعمال می‌کند. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">Q</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">K</code> را ثابت نگه دارید و فقط <bdi dir="ltr">Value</bdi> موقعیت آخر را تغییر دهید. خروجی دو موقعیت اول باید ثابت بماند؛ چرا موقعیت آخر می‌تواند تغییر کند؟ اینجا خروجی <bdi dir="ltr">Attention</bdi> را می‌بینیم، نه خود جدول وزن‌ها را.</p>
</div>

In [ ]:
from torch.nn import functional as F
allowed = torch.tensor([[True,False,False],[True,True,False],[True,True,True]])
q,k = torch.eye(3),torch.eye(3)
v = torch.tensor([[1.,2.],[3.,4.],[5.,6.]])
changed_v = v.clone()
changed_v[-1] += 1000
before = F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.)
after = F.scaled_dot_product_attention(q,k,changed_v,is_causal=True,dropout_p=0.)
print('allowed keys per query:',allowed.sum(-1))
print('output change per query:',(after-before).abs().amax(-1))
torch.testing.assert_close(before[:2],after[:2])

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">نسخهٔ خراب امتیاز ممنوع را صفر می‌کند. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">repair_mask(scores)</code> را اصلاح کنید؛ سطر بدون <bdi dir="ltr">Key</bdi> مجاز نسازید.</p>
</div>

In [ ]:
wrong = scores.masked_fill(~allowed,0.).softmax(-1)
print('future mass in first row:',wrong[0,1:].sum().item())
strict = torch.ones(3,3,dtype=torch.bool).tril(-1)
print('empty first row is finite:',torch.isfinite(scores.masked_fill(~strict,float('-inf')).softmax(-1)[0]).all().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def repair_mask(scores):
    # TODO
    return None

In [ ]:
def test_repair():
    result = repair_mask(scores)
    if result is None: return False
    assert torch.isfinite(result).all()
    assert result[0,0].item() == 1.
    assert torch.count_nonzero(result.triu(1)) == 0
    expected = scores.masked_fill(~torch.ones(3,3,dtype=torch.bool).tril(),float('-inf')).softmax(-1)
    torch.testing.assert_close(result,expected)
    torch.testing.assert_close(repair_mask(torch.zeros(2,2)),torch.tensor([[1.,0.],[0.5,0.5]]))
    torch.testing.assert_close(repair_mask(torch.zeros(1,1)),torch.ones(1,1))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><bdi dir="ltr">v3</bdi> همین محدودیت را به <bdi dir="ltr">v2</bdi> اضافه می‌کند؛ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">CausalSelfAttention</code> نهایی <bdi dir="ltr">Mask</bdi> را به‌صورت <bdi dir="ltr">Buffer</bdi> نگه می‌دارد. وجود <bdi dir="ltr">Token</bdi> آینده در <bdi dir="ltr">Batch</bdi> با مجازبودن دسترسی به آن یکی نیست.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">چرا قطر جدول مجاز است ولی ستون بعد از آن می‌تواند پاسخ هدف فعلی را لو بدهد؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-03/33-mask.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/33-mask.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>